# 00 — Environment Setup & GPU Check

Run this notebook first after creating your OpenShift AI Workbench.
It verifies GPU access, installs dependencies, and runs a smoke test.

**OpenShift AI Workbench settings:**
- Image: PyTorch (CUDA)
- Container size: Large (8+ CPU, 32+ GB RAM)
- Accelerator: NVIDIA GPU x1
- Persistent storage: >= 30 GB

## 1. GPU Verification

In [1]:
!nvidia-smi

Tue Jul 28 17:31:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   38C    P8             28W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:             {gpu_name}")
    print(f"VRAM:            {vram_gb:.1f} GB")

    if vram_gb < 20:
        print("\n⚠  VRAM < 20 GB — Qwen3-8B in fp16 may not fit.")
        print("   The experiment notebooks will use 4-bit quantization (BitsAndBytesConfig).")
    else:
        print("\n✓  Enough VRAM for Qwen3-8B in fp16.")
else:
    raise RuntimeError(
        "No CUDA GPU detected. Make sure your Workbench has a GPU accelerator assigned."
    )

PyTorch version: 2.10.0
CUDA available:  True
GPU:             NVIDIA A10G
VRAM:            23.7 GB

✓  Enough VRAM for Qwen3-8B in fp16.


## 2. Environment & Dependencies

In [3]:
import os
os.environ["HF_TOKEN"] = "YOUR_TOKEN_HERE"

Re-run the cell below if the pod was recreated. Pip will skip already-installed packages.

In [4]:
!pip install git+https://github.com/NVIDIA/kvpress.git@v0.5.4 --no-deps
!pip install vllm==0.18.0 transformers==4.57.6 datasets==3.6.0 \
    rouge-score==0.1.2 matplotlib==3.10.8 pandas==2.3.3 \
    bitsandbytes==0.49.2 accelerate==1.12.0 fire==0.7.1

Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.4/cuda13.0-ubi9/simple/
  Cloning https://github.com/NVIDIA/kvpress.git (to revision v0.5.4) to /tmp/pip-req-build-bkg0pg9o
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/kvpress.git /tmp/pip-req-build-bkg0pg9o
  Running command git checkout -q 6d965557a5b9f0201a2301b23c454473dd681d0d
  Resolved https://github.com/NVIDIA/kvpress.git to commit 6d965557a5b9f0201a2301b23c454473dd681d0d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.4/cuda13.0-ubi9/simple/


## 3. Verify Imports

In [5]:
from importlib.metadata import version
import kvpress, vllm

for pkg in ["kvpress", "vllm", "transformers", "datasets", "rouge-score", "matplotlib", "pandas"]:
    print(f"{pkg:20s} {version(pkg)}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kvpress              0.5.4
vllm                 0.18.0+rhaiv.4
transformers         4.57.6
datasets             3.6.0
rouge-score          0.1.2
matplotlib           3.10.8
pandas               2.3.3


## 4. Verify kvpress Press Types

In [6]:
from kvpress import KeyDiffPress, BlockPress, PrefillDecodingPress, CompressionRatioDecodingPress

press = PrefillDecodingPress(
    prefilling_press=BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128),
    decoding_press=CompressionRatioDecodingPress(
        base_press=KeyDiffPress(), target_compression_ratio=0.5,
    ),
)

print(f"PrefillDecodingPress: {press}")
print(f"  prefilling: {press.prefilling_press}")
print(f"  decoding:   {press.decoding_press}")
print("\n✓  Press types instantiated successfully.")

PrefillDecodingPress: PrefillDecodingPress(prefilling_press=BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128), decoding_press=CompressionRatioDecodingPress(base_press=KeyDiffPress(compression_ratio=0.0), compression_interval=512, target_size=1, hidden_states_buffer_size=256, target_compression_ratio=0.5))
  prefilling: BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128)
  decoding:   CompressionRatioDecodingPress(base_press=KeyDiffPress(compression_ratio=0.0), compression_interval=512, target_size=1, hidden_states_buffer_size=256, target_compression_ratio=0.5)

✓  Press types instantiated successfully.


## 5. Smoke Test — CUDA Tensor Round-Trip

In [7]:
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
y = x @ x.T
print(f"Matrix multiply on GPU OK — result shape: {y.shape}, dtype: {y.dtype}")
del x, y
torch.cuda.empty_cache()

Matrix multiply on GPU OK — result shape: torch.Size([1024, 1024]), dtype: torch.float16


## 6. Dataset Access Check

In [8]:
from datasets import load_dataset

ds = load_dataset("alessiodevoto/paul_graham_essays", split="test")
print(f"Paul Graham essays loaded: {len(ds)} rows")
print(f"Columns: {ds.column_names}")
print(f"Context length: {len(ds[0]['context'].split())} words")
print(f"Needle: {ds[0]['needle'][:80]}...")
print(f"Question: {ds[0]['question']}")

Paul Graham essays loaded: 1 rows
Columns: ['context', 'needle', 'question', 'answer_prefix', 'max_new_tokens']
Context length: 525450 words
Needle: 

Remember, the best thing to do in San Francisco is eat a sandwich and sit in D...
Question: 

Question: Based on the content of the book, what is the best thing to do in San Francisco?


## Done

If all cells above ran without errors, your environment is ready.  
Proceed to:
- `01_kvpress_niah.ipynb` — KeyDiffPress NIAH experiments
- `02_vllm_niah.ipynb` — vLLM baseline NIAH experiments